# Host SLM Router trên Google Colab

Notebook này deploy `hanoi-weather-router` (Qwen3-4B fine-tuned, GGUF Q4_K_M) lên Colab GPU, expose public URL qua cloudflared tunnel để FastAPI/worker local kết nối được.

## Trước khi chạy

1. **Runtime → Change runtime type → GPU (T4 miễn phí là đủ)**.
2. Chạy lần lượt từng cell. Cell cuối sẽ in ra `https://xxxxxx.trycloudflare.com` — copy URL này để config Docker stack.

## Lưu ý

- Colab free tier disconnect sau ~90 phút idle hoặc ~12h runtime. Dùng cho demo/testing, không dùng production.
- URL `trycloudflare.com` thay đổi mỗi lần chạy lại tunnel → cần update `OLLAMA_BASE_URL` ở local sau mỗi lần restart.

## 1. Cài Ollama

In [1]:
# Install script của Ollama dùng zstd để giải nén — Colab base image chưa có sẵn
!apt-get -qq install -y zstd

!curl -fsSL https://ollama.com/install.sh | sh

# Verify: phải thấy 'ollama version x.y.z'. Nếu thiếu dòng này → install fail, dừng lại.
!/usr/local/bin/ollama --version

Selecting previously unselected package zstd.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## 2. Cài cloudflared (tunnel public, không cần auth)

In [2]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

cloudflared version 2026.3.0 (built 2026-03-09-14:08 UTC)


## 3. Start Ollama server (background)

Set `OLLAMA_HOST=0.0.0.0:11434` để lắng nghe public (cloudflared sẽ tunnel tới port này).

In [3]:
import os
import subprocess
import time

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"

# Đảm bảo /usr/local/bin trong PATH (cho các cell sau gọi `!ollama ...`)
if "/usr/local/bin" not in os.environ["PATH"]:
    os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]

# Dùng absolute path để subprocess.Popen không phụ thuộc PATH
subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    env=os.environ.copy(),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)

# Verify server đang chạy
!curl -s http://localhost:11434/api/tags
print("\n Ollama server ready")

{"models":[]}
 Ollama server ready


## 4. Tạo model `hanoi-weather-router` từ GGUF của HF

Modelfile được viết inline — source trỏ tới repo GGUF trên HuggingFace. Ollama sẽ tự pull GGUF (default Q4_K_M, ~2.5GB). System prompt + params giữ nguyên như `model/router/Modelfile.v4` của dự án.

**Lưu ý**: nếu repo HF có nhiều quantization, có thể chỉ định cụ thể: `FROM hf.co/daredevil467/hanoi-router-qwen3-4b-v6-gguf:Q4_K_M`.

In [4]:
modelfile = r'''FROM hf.co/daredevil467/hanoi-router-qwen3-4b-v6-gguf

TEMPLATE """<|im_start|>system
{{ .System }}<|im_end|>
<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
"""

PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
PARAMETER temperature 0
PARAMETER top_p 1.0
PARAMETER repeat_penalty 1.0

SYSTEM """Phân loại intent và scope cho câu hỏi thời tiết Hà Nội. Trả về JSON.

## Intents:
- current_weather: thời tiết NGAY LÚC NÀY (nhiệt độ, trời nắng/mưa, chung chung)
- hourly_forecast: diễn biến CHI TIẾT THEO GIỜ trong ngày (không chỉ hỏi mưa)
- daily_forecast: dự báo NHIỀU NGÀY tới (3 ngày, tuần tới, cuối tuần)
- weather_overview: TỔNG QUAN, tóm tắt thời tiết hôm nay/ngày mai (không hỏi thông số cụ thể)
- rain_query: hỏi CÓ MƯA KHÔNG, xác suất mưa, mưa bao lâu/lúc nào tạnh
- temperature_query: hỏi CỤ THỂ VỀ NHIỆT ĐỘ (bao nhiêu độ, nóng/lạnh)
- wind_query: hỏi CỤ THỂ VỀ GIÓ (gió mạnh không, hướng gió, tốc độ gió)
- humidity_fog_query: hỏi về ĐỘ ẨM, SƯƠNG MÙ, sương muối
- historical_weather: thời tiết NGÀY/TUẦN TRƯỚC, dữ liệu QUÁ KHỨ
- location_comparison: SO SÁNH thời tiết giữa các quận/phường/địa điểm
- activity_weather: thời tiết PHÙ HỢP ĐỂ LÀM hoạt động X không (chạy bộ, picnic)
- expert_weather_param: thông số KỸ THUẬT chuyên sâu (áp suất, UV, điểm sương, tầm nhìn)
- weather_alert: CẢNH BÁO nguy hiểm: bão/áp thấp, ngập, giông/lốc, rét hại, nắng nóng cực đoan
- seasonal_context: SO SÁNH với hôm qua/tuần trước, xu hướng, bất thường theo MÙA
- smalltalk_weather: chào hỏi, ngoài phạm vi, câu hỏi không liên quan thời tiết

## Anti-confusion rules:
- bây giờ/bay gio = thời điểm hiện tại -> current_weather (KHÔNG phải wind_query)
- gió/gió mạnh/tốc độ gió = yếu tố gió -> wind_query
- bão/lũ/cảnh báo -> weather_alert (KHÔNG phải rain_query)

## Scopes:
- city: toàn Hà Nội hoặc không nói rõ địa điểm
- district: quận/huyện hoặc địa điểm nổi tiếng (Hồ Gươm->Hoàn Kiếm, Lăng Bác->Ba Đình, Nội Bài->Sóc Sơn...)
- ward: phường/xã

## Output:
{"intent":"...","scope":"...","confidence":0.9}
Thêm rewritten_query nếu có context và câu thiếu địa điểm."""
'''

with open("/content/Modelfile", "w") as f:
    f.write(modelfile)

print("Modelfile written to /content/Modelfile")
!cat /content/Modelfile | head -15

Modelfile written to /content/Modelfile
FROM hf.co/daredevil467/hanoi-router-qwen3-4b-v6-gguf

TEMPLATE """<|im_start|>system
{{ .System }}<|im_end|>
<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
"""

PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
PARAMETER temperature 0
PARAMETER top_p 1.0
PARAMETER repeat_penalty 1.0



In [5]:
# Pull + compile model — mất 2-5 phút lần đầu (download GGUF ~2.5GB)
!ollama create hanoi-weather-router -f /content/Modelfile
!ollama list


NAME                                                       ID              SIZE      MODIFIED               
hf.co/daredevil467/hanoi-router-qwen3-4b-v6-gguf:latest    53c68a47709d    2.5 GB    Less than a second ago    
hanoi-weather-router:latest                                91cb5fcac239    2.5 GB    Less than a second ago    


## 5. Smoke test: classify 1 câu hỏi

In [6]:
import requests

# Lần đầu Ollama load model 4B vào VRAM mất ~30-90s trên T4. Inference sau đó < 1s.
# Dùng timeout lớn (300s) để tránh ReadTimeout. Tham số keep_alive giữ model trong VRAM 30 phút.
print("⏳ Đang classify... (lần đầu có thể 30-90s do load model)")

resp = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "hanoi-weather-router",
        "prompt": "Thời tiết Cầu Giấy hôm nay thế nào?",
        "stream": False,
        "format": "json",
        "keep_alive": "30m",
    },
    timeout=300,
)
print("Response:", resp.json()["response"])
print(f"Total time: {resp.json().get('total_duration', 0) / 1e9:.1f}s")

⏳ Đang classify... (lần đầu có thể 30-90s do load model)
Response: {"intent": "weather_overview", "scope": "district", "confidence": 0.9}
Total time: 25.3s


## 6. Start cloudflared tunnel → lấy public URL

Cell này khởi tunnel + extract URL từ log. Copy URL để config Docker stack.

In [7]:
import re
import subprocess
import threading
import time

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
pattern = re.compile(r"https://[\w-]+\.trycloudflare\.com")

# Đọc log để tìm URL (cloudflared in URL sau ~3-5s)
start = time.time()
while time.time() - start < 30 and public_url is None:
    line = proc.stdout.readline()
    if not line:
        continue
    print(line, end="")
    m = pattern.search(line)
    if m:
        public_url = m.group(0)

# Tiếp tục drain stdout để tunnel không block
def _drain():
    for _ in proc.stdout:
        pass

threading.Thread(target=_drain, daemon=True).start()

if not public_url:
    raise RuntimeError("Không tìm được public URL từ cloudflared log")

print("\n" + "=" * 60)
print(f"🟢 PUBLIC URL: {public_url}")
print("=" * 60)
print("\n📋 Thêm vào .env của project local:")
print(f"OLLAMA_BASE_URL={public_url}")
print("OLLAMA_MODEL=hanoi-weather-router")
print("USE_SLM_ROUTER=true")
print("\nSau đó: docker compose restart api worker")

2026-04-17T14:00:01Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-17T14:00:01Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-17T14:00:10Z INF +--------------------------------------------------------------------------------------------+
2026-04-17T14:00:10Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-17T14:00:10Z INF |  https://wings-equations-cache-con.trycloudflare.com  

## 7. Test public URL (optional)

Kiểm tra tunnel hoạt động từ ngoài Colab.

In [9]:
import requests

# Model đã được warmup ở Cell 5 → inference nhanh (< 2s)
r = requests.post(
    f"{public_url}/api/generate",
    json={
        "model": "hanoi-weather-router",
        "prompt": "Ngày mai Đống Đa có mưa không?",
        "stream": False,
        "format": "json",
        "keep_alive": "30m",
    },
    timeout=120,
)
print(r.status_code)
print(r.json().get("response"))

200
{"intent": "rain_query", "scope": "district", "confidence": 0.9}


## Giữ runtime sống

Colab free tier kill runtime sau ~90 phút idle. Để giữ runtime, chạy cell dưới (sleep vô hạn, in dot mỗi 60s). Dùng Ctrl+C / Interrupt runtime để dừng.

In [10]:
import time

print("Keep-alive loop started. Interrupt để dừng.")
try:
    while True:
        time.sleep(60)
        print(".", end="", flush=True)
except KeyboardInterrupt:
    print("\nStopped.")

Keep-alive loop started. Interrupt để dừng.
......
Stopped.
